# Comparison with pyvinecopulib

Compact MLE-only comparison between `pyscarcopula` and `pyvinecopulib` on S&P 500 pseudo-observations. The benchmark uses fixed Gumbel pair-copulas and a shallow R-vine truncation level to keep the focus on fit, likelihood, sampling, and selected structure differences.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyvinecopulib as pv
from scipy import stats

from pyscarcopula._utils import pobs
from pyscarcopula import GumbelCopula, RVineCopula

## Data and controls

In [2]:
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('..') / 'data'

DIMS = [10, 50, 100]
TRUNCATION_LEVEL = 2
N_SAMPLE_BY_DIM = {10: 5000, 50: 2000, 100: 1000}
SEED = 2026

PV_FAMILY = pv.BicopFamily.gumbel
PV_ROTATION = 0

prices = pd.read_csv(DATA_DIR / 'sp500_yfinance_2020_prices.csv', parse_dates=['Date']).set_index('Date')
prices = prices.apply(pd.to_numeric, errors='coerce')
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices))).ffill().dropna()
returns = np.log(prices / prices.shift(1)).dropna()
assets = list(returns.columns)
available_dims = [d for d in DIMS if d <= len(assets)]

print(f'Prices: {prices.shape[0]} rows, {prices.shape[1]} assets')
print(f'Returns: {returns.shape[0]} rows, {returns.shape[1]} assets')
print('Benchmark dimensions:', available_dims)


def dataset_for_dim(d):
    cols = assets[:d]
    return np.asarray(pobs(returns[cols].values), dtype=np.float64), cols

Prices: 1582 rows, 105 assets
Returns: 1581 rows, 105 assets
Benchmark dimensions: [10, 50, 100]


## Helpers

In [3]:
def as_fortran(u):
    return np.asarray(u, dtype=np.float64, order='F')


def timed(fn):
    t0 = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t0


def ps_fixed_copulas(d):
    return [[(GumbelCopula, 0) for _ in range(d - 1 - tree)] for tree in range(d - 1)]


def pv_fixed_pair_copulas(d):
    n_levels = min(TRUNCATION_LEVEL, d - 1)
    return [[pv.Bicop(PV_FAMILY, PV_ROTATION) for _ in range(d - 1 - tree)] for tree in range(n_levels)]


def fit_ps_vine(u):
    vine = RVineCopula(candidates=[GumbelCopula], allow_rotations=False)
    return vine.fit(u, method='mle', truncation_level=TRUNCATION_LEVEL, copulas=ps_fixed_copulas(u.shape[1]))


def fit_pv_bicop(u):
    bicop = pv.Bicop(PV_FAMILY, PV_ROTATION)
    bicop.fit(as_fortran(u), controls=pv.FitControlsBicop(preselect_families=False))
    return bicop


def fit_pv_vine(u):
    controls = pv.FitControlsVinecop(
        family_set=[PV_FAMILY],
        allow_rotations=False,
        trunc_lvl=TRUNCATION_LEVEL,
        num_threads=1,
        preselect_families=False,
    )
    selected = pv.Vinecop.from_data(as_fortran(u), controls=controls)
    model = pv.Vinecop.from_structure(
        structure=selected.structure,
        pair_copulas=pv_fixed_pair_copulas(u.shape[1]),
    )
    model.fit(as_fortran(u), controls=pv.FitControlsBicop(preselect_families=False), num_threads=1)
    return model


def corr_frobenius(a, b):
    ca = np.corrcoef(a, rowvar=False)
    cb = np.corrcoef(b, rowvar=False)
    return float(np.linalg.norm(ca - cb, ord='fro') / ca.shape[0])


def marginal_ks_mean(x, y):
    return float(np.mean([stats.ks_2samp(x[:, j], y[:, j]).statistic for j in range(x.shape[1])]))


def ps_order(vine):
    return [int(vine.matrix[vine.d - 1 - col, col]) for col in range(vine.d)]


def pv_order(vine):
    return [int(x) - 1 for x in vine.structure.order]


def ps_tree_edges(vine, tree):
    return {
        (tuple(sorted(int(v) for v in conditioned)), tuple(sorted(int(v) for v in conditioning)))
        for conditioned, conditioning in vine.trees[tree]
    }


def pv_tree_edges(vine, tree):
    order = pv_order(vine)
    edges = set()
    for edge_idx in range(vine.dim - 1 - tree):
        conditioned = tuple(sorted((
            int(vine.structure.struct_array(tree, edge_idx, False)) - 1,
            order[edge_idx],
        )))
        conditioning = tuple(sorted(
            int(vine.structure.struct_array(k, edge_idx, False)) - 1 for k in range(tree)
        ))
        edges.add((conditioned, conditioning))
    return edges

## Bivariate fit, density, and sampling

In [4]:
u2, cols2 = dataset_for_dim(2)

ps_bi = GumbelCopula(rotate=0)
ps_result, ps_fit_t = timed(lambda: ps_bi.fit(u2, method='mle'))
pv_bi, pv_fit_t = timed(lambda: fit_pv_bicop(u2))

ps_pdf, ps_pdf_t = timed(lambda: np.exp(ps_bi.log_pdf(
    u2[:, 0], u2[:, 1], np.full(len(u2), ps_result.copula_param)
)))
pv_pdf, pv_pdf_t = timed(lambda: pv_bi.pdf(as_fortran(u2)))

ps_sample, ps_sample_t = timed(lambda: ps_bi.sample(
    5000, r=ps_result.copula_param, rng=np.random.default_rng(SEED)
))
pv_sample, pv_sample_t = timed(lambda: pv_bi.simulate(5000, seeds=[SEED]))

pv_param = float(np.asarray(pv_bi.parameters, dtype=float).ravel()[0])
pd.DataFrame([{
    'assets': ', '.join(cols2),
    'param_pyscarcopula': float(ps_result.copula_param),
    'param_pyvinecopulib': pv_param,
    'param_abs_diff': abs(float(ps_result.copula_param) - pv_param),
    'loglik_pyscarcopula': ps_result.log_likelihood,
    'loglik_pyvinecopulib': pv_bi.loglik(as_fortran(u2)),
    'pdf_max_abs_diff': float(np.max(np.abs(ps_pdf - pv_pdf))),
    'fit_time_pyscarcopula_s': ps_fit_t,
    'fit_time_pyvinecopulib_s': pv_fit_t,
    'pdf_time_pyscarcopula_s': ps_pdf_t,
    'pdf_time_pyvinecopulib_s': pv_pdf_t,
    'sample_time_pyscarcopula_s': ps_sample_t,
    'sample_time_pyvinecopulib_s': pv_sample_t,
    'sample_ks_dim0': stats.ks_2samp(ps_sample[:, 0], pv_sample[:, 0]).statistic,
    'sample_ks_dim1': stats.ks_2samp(ps_sample[:, 1], pv_sample[:, 1]).statistic,
}]).T

,0
assets,"AAPL, ABBV"
param_pyscarcopula,1.152249
param_pyvinecopulib,1.152249
param_abs_diff,0.0
loglik_pyscarcopula,44.986205
loglik_pyvinecopulib,44.986205
pdf_max_abs_diff,0.000086
fit_time_pyscarcopula_s,0.691748
fit_time_pyvinecopulib_s,0.003798
pdf_time_pyscarcopula_s,0.000214


## R-vine benchmark

In [5]:
rows = []
models = {}

for d in available_dims:
    u_d, cols_d = dataset_for_dim(d)
    n_sample = N_SAMPLE_BY_DIM.get(d, 1000)

    ps_vine, ps_fit_t = timed(lambda u=u_d: fit_ps_vine(u))
    pv_vine, pv_fit_t = timed(lambda u=u_d: fit_pv_vine(u))
    ps_ll, ps_ll_t = timed(lambda u=u_d, m=ps_vine: m.log_likelihood(u))
    pv_ll, pv_ll_t = timed(lambda u=u_d, m=pv_vine: m.loglik(as_fortran(u)))
    ps_samp, ps_sample_t = timed(lambda m=ps_vine, d=d: m.sample(n_sample, rng=np.random.default_rng(SEED + d)))
    pv_samp, pv_sample_t = timed(lambda m=pv_vine, d=d: m.simulate(n_sample, seeds=[SEED + d]))

    rows.append({
        'd': d,
        'n_obs': len(u_d),
        'n_sample': n_sample,
        'fit_time_pyscarcopula_s': ps_fit_t,
        'fit_time_pyvinecopulib_s': pv_fit_t,
        'loglik_pyscarcopula': ps_ll,
        'loglik_pyvinecopulib': pv_ll,
        'loglik_diff_per_obs': (ps_ll - pv_ll) / len(u_d),
        'loglik_time_pyscarcopula_s': ps_ll_t,
        'loglik_time_pyvinecopulib_s': pv_ll_t,
        'sample_time_pyscarcopula_s': ps_sample_t,
        'sample_time_pyvinecopulib_s': pv_sample_t,
        'ps_sample_corr_fro_to_data': corr_frobenius(ps_samp, u_d),
        'pv_sample_corr_fro_to_data': corr_frobenius(pv_samp, u_d),
        'sample_ks_mean_pyscarcopula': marginal_ks_mean(ps_samp, u_d),
        'sample_ks_mean_pyvinecopulib': marginal_ks_mean(pv_samp, u_d),
    })
    models[d] = {'u': u_d, 'cols': cols_d, 'ps_vine': ps_vine, 'pv_vine': pv_vine}

benchmark = pd.DataFrame(rows)
benchmark

,d,n_obs,n_sample,fit_time_pyscarcopula_s,fit_time_pyvinecopulib_s,loglik_pyscarcopula,loglik_pyvinecopulib,loglik_diff_per_obs,loglik_time_pyscarcopula_s,loglik_time_pyvinecopulib_s,sample_time_pyscarcopula_s,sample_time_pyvinecopulib_s,ps_sample_corr_fro_to_data,pv_sample_corr_fro_to_data,sample_ks_mean_pyscarcopula,sample_ks_mean_pyvinecopulib
0,10,1581,5000,0.344049,0.180249,3316.065595,3316.065585,6.127556e-09,0.007371,0.013559,0.690212,0.041023,0.092303,0.095716,0.011622,0.012963
1,50,1581,2000,0.882329,1.472423,26278.553975,26277.801713,4.758144e-04,0.039307,0.067799,0.088845,0.090104,0.210239,0.208214,0.018651,0.020986
2,100,1581,1000,3.053127,4.051809,55664.331027,55661.176519,1.995261e-03,0.081868,0.139523,0.129664,0.094609,0.231060,0.205272,0.026151,0.027540


## Structure snapshot (`d=10`)

In [6]:
m10 = models[10]
ps_v10 = m10['ps_vine']
pv_v10 = m10['pv_vine']
cols10 = m10['cols']
active_levels = min(TRUNCATION_LEVEL, ps_v10.d - 1)

structure_rows = []
for tree in range(active_levels):
    ps_edges = ps_tree_edges(ps_v10, tree)
    pv_edges = pv_tree_edges(pv_v10, tree)
    union = ps_edges | pv_edges
    structure_rows.append({
        'tree': tree,
        'edges_pyscarcopula': len(ps_edges),
        'edges_pyvinecopulib': len(pv_edges),
        'common_edges': len(ps_edges & pv_edges),
        'jaccard': len(ps_edges & pv_edges) / len(union) if union else 1.0,
    })

print('pyscarcopula order:', [cols10[i] for i in ps_order(ps_v10)])
print('pyvinecopulib order:', [cols10[i] for i in pv_order(pv_v10)])
pd.DataFrame(structure_rows)

pyscarcopula order: ['ABBV', 'AMD', 'AMAT', 'ADI', 'AAPL', 'ADBE', 'ACN', 'ADP', 'ABT', 'AMGN']
pyvinecopulib order: ['AMD', 'AMAT', 'ADI', 'AAPL', 'ADBE', 'ACN', 'ADP', 'ABBV', 'ABT', 'AMGN']


,tree,edges_pyscarcopula,edges_pyvinecopulib,common_edges,jaccard
0,0,9,9,9,1.0
1,1,8,8,8,1.0


## Notes

The libraries use different R-vine matrix conventions, so this notebook compares decoded order and active edges instead of raw matrices. Timings are simple wall-clock measurements from one run; repeat the cells when you need stable benchmark numbers.